# 📊 Market Research Synthesis: LLM-Driven Qualitative Analysis of Audience Comments
### *Automated Consumer Insights, Sentiment Extraction, and Audience Intent Analysis via Gemini 3.1 Flash-Lite*

---

## 📋 Abstract & Analytical Methodology

This notebook presents an end-to-end qualitative research pipeline that extracts actionable market research insights, customer pain points, feature/topic requests, and sentiment trends from top YouTube video comments. Operating on exported comment payloads generated by preceding extraction workflows (`youtube_video_comments.ipynb`), this pipeline leverages **Gemini 3.1 Flash-Lite** (`gemini-3.1-flash-lite`) via the official `google-genai` SDK to synthesize vast unstructured viewer feedback into structured qualitative artifacts.

### 🔬 Input Dependencies & Key Artifacts
- **Input Payload**: `{Channel_Name}_top_comments_10_recent_videos.json` (or `.csv`) located in Google Drive (`/content/drive/MyDrive/YouTube_Analytics/`) or local fallbacks (`data/`, `analysis/`, `./`).
- **Primary LLM Engine**: `google-genai` SDK targeting model `gemini-3.1-flash-lite` with automated retry mechanics (up to 3 attempts with exponential backoff $T_{wait} = 2 \times \text{attempt}$ seconds).
- **Persistence & Caching**: Disk-backed persistent generation cache (`llm_generation_cache.json`) synchronized across Google Drive and local storage to prevent duplicate API requests upon execution restarts.
- **Exported Artifacts**: 
  1. Formatted Market Research Markdown Report (`{Channel_Name}_market_research_summary.md`)
  2. Structured JSON Market Research Dataset (`{Channel_Name}_market_research_summary.json`)

### 📐 Mathematical & Pipeline Notation
Given a set of recent videos $V = \{v_1, v_2, \dots, v_M\}$ and associated top comment collections $C(v_i) = \{c_{i,1}, c_{i,2}, \dots, c_{i,K}\}$, the LLM transformation function $f_{\text{LLM}}: C(v_i) \to \mathcal{S}_i$ maps unstructured feedback to a structured qualitative vector $\mathcal{S}_i = \langle \text{PainPoints}_i, \text{Requests}_i, \text{Sentiment}_i, \text{KeyInsights}_i \rangle$. A global synthesis operator $\Omega: \{\mathcal{S}_1, \dots, \mathcal{S}_M\} \to \mathcal{M}_{channel}$ produces the aggregate channel-level market research report.

## ⚙️ 1. Configuration, Dependencies & Environment Setup

In this section, we install required packages (`google-genai`, `pandas`, `matplotlib`, `seaborn`), specify model hyperparameters, configure the API key detection sequence (checking Colab User Data secrets first, then environment variables), and establish global runtime parameters.

In [ ]:
# Install required dependencies
!pip install -q google-genai pandas matplotlib seaborn

import os
import sys
import re
import time
import json
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Any, Optional

# Attempt to import google-genai SDK
try:
    from google import genai
    from google.genai import types
except ImportError:
    raise ImportError("The 'google-genai' package is required. Please install it using 'pip install google-genai'.")

# Global Configuration
CONFIG = {
    "MODEL_NAME": "gemini-3.1-flash-lite",
    "MAX_RETRIES": 3,
    "BACKOFF_FACTOR": 2,
    "TEMPERATURE": 0.2,
    "TOP_P": 0.95,
    "CHANNEL_IDENTIFIER": "@josemillanastrologohumanista",
    "DRIVE_OUTPUT_DIR": "/content/drive/MyDrive/YouTube_Analytics",
    "LOCAL_FALLBACK_DIRS": ["data", "analysis", "."],
    "CACHE_FILE_NAME": "llm_generation_cache.json",
    "MOUNT_DRIVE": True,
    "USE_CACHE": True
}

def get_api_key() -> str:
    """
    Retrieves the Google API Key following the strict fallback sequence:
    1. Google Colab User Data secrets ('GOOGLE_API_KEY')
    2. Environment Variable 'GOOGLE_API_KEY'
    3. Environment Variable 'GEMINI_API_KEY'
    """
    api_key = None
    # 1. Check Colab User Data
    try:
        from google.colab import userdata
        api_key = userdata.get("GOOGLE_API_KEY")
        if api_key:
            print("🔑 API Key successfully retrieved from Google Colab Secrets.")
            return api_key
    except Exception:
        pass
    
    # 2. Check environment variables
    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if api_key:
        print("🔑 API Key successfully retrieved from Environment Variables.")
        return api_key
    
    print("⚠️ Warning: GOOGLE_API_KEY not found in Colab Secrets or Environment Variables.")
    return ""

API_KEY = get_api_key()
print(f"Target Model: {CONFIG['MODEL_NAME']}")
print(f"Max Retries: {CONFIG['MAX_RETRIES']} | Backoff Factor: {CONFIG['BACKOFF_FACTOR']}s")

## 💾 2. Mount Google Drive & Storage Initialization

This cell mounts Google Drive at `/content/drive` with a robust local fallback hierarchy (`/content/drive/MyDrive/YouTube_Analytics/`, `data/`, `analysis/`, `./`). This ensures seamless cross-platform execution whether running directly in Google Colab or locally.

In [ ]:
ACTIVE_OUTPUT_DIR = "."

if CONFIG["MOUNT_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(CONFIG["DRIVE_OUTPUT_DIR"], exist_ok=True)
        ACTIVE_OUTPUT_DIR = CONFIG["DRIVE_OUTPUT_DIR"]
        print(f"✅ Google Drive mounted successfully! Output directory: {ACTIVE_OUTPUT_DIR}")
    except ImportError:
        print("ℹ️ google.colab not available. Operating in local environment.")
        ACTIVE_OUTPUT_DIR = "data" if os.path.exists("data") else "."
    except Exception as e:
        print(f"⚠️ Mount failed or skipped: {e}. Falling back to local directory.")
        ACTIVE_OUTPUT_DIR = "."
else:
    ACTIVE_OUTPUT_DIR = "."

print(f"📁 Active Primary Storage Path: {os.path.abspath(ACTIVE_OUTPUT_DIR)}")

## 🔄 3. Persistent Disk Caching & LLM Invocation Engine with Retry Logic

In accordance with project design directives, this section implements:
1. **Persistent Generation Caching**: Saves and retrieves LLM response prompts to/from `llm_generation_cache.json` stored in Google Drive and local storage paths.
2. **Strict Retry Loop**: Attempts LLM requests up to **3 times** upon API or network failure using exponential backoff ($T_{backoff} = 2 \times \text{attempt}$ seconds).
3. **Explicit Error Handling**: If all retries fail, a `RuntimeError` is raised rather than populating outputs with synthetic or placeholder text.

In [ ]:
class PersistentLLMCache:
    """
    Manages a persistent JSON disk cache for LLM responses across Google Drive
    and local fallback paths to avoid redundant API queries.
    """
    def __init__(self, primary_dir: str, fallback_dirs: List[str], filename: str):
        self.filename = filename
        self.search_paths = [primary_dir] + fallback_dirs
        self.cache_file_path = os.path.join(primary_dir, filename)
        self.cache: Dict[str, Any] = self._load_cache()

    def _load_cache(self) -> Dict[str, Any]:
        for directory in self.search_paths:
            path = os.path.join(directory, self.filename)
            if os.path.exists(path):
                try:
                    with open(path, "r", encoding="utf-8") as f:
                        data = json.load(f)
                        print(f"📦 Loaded persistent LLM cache from: {path} ({len(data)} entries)")
                        return data
                except Exception as e:
                    print(f"⚠️ Could not read cache file at {path}: {e}")
        print("🆕 Initializing new empty persistent LLM cache.")
        return {}

    def get(self, key: str) -> Optional[str]:
        return self.cache.get(key)

    def set(self, key: str, value: str):
        self.cache[key] = value
        self._save_cache()

    def _save_cache(self):
        for directory in self.search_paths:
            try:
                os.makedirs(directory, exist_ok=True)
                path = os.path.join(directory, self.filename)
                with open(path, "w", encoding="utf-8") as f:
                    json.dump(self.cache, f, indent=2, ensure_ascii=False)
            except Exception as e:
                pass

# Initialize persistent cache
llm_cache = PersistentLLMCache(
    primary_dir=ACTIVE_OUTPUT_DIR,
    fallback_dirs=CONFIG["LOCAL_FALLBACK_DIRS"],
    filename=CONFIG["CACHE_FILE_NAME"]
)

def call_gemini_with_retry(prompt: str, cache_key: Optional[str] = None) -> str:
    """
    Executes an LLM call using the google-genai SDK targeting gemini-3.1-flash-lite.
    Implements persistent disk caching and an explicit retry loop (max 3 retries).
    Throws RuntimeError if all attempts fail.
    """
    if CONFIG["USE_CACHE"] and cache_key:
        cached_response = llm_cache.get(cache_key)
        if cached_response:
            print(f"⚡ Cache hit for key: {cache_key[:40]}...")
            return cached_response

    if not API_KEY:
        raise RuntimeError("Cannot call Gemini API: API Key is missing or invalid.")

    client = genai.Client(api_key=API_KEY)
    max_retries = CONFIG["MAX_RETRIES"]
    backoff_factor = CONFIG["BACKOFF_FACTOR"]
    last_exception = None

    for attempt in range(1, max_retries + 1):
        try:
            print(f"🤖 Invoking Gemini API ({CONFIG['MODEL_NAME']}), Attempt {attempt}/{max_retries}...")
            response = client.models.generate_content(
                model=CONFIG["MODEL_NAME"],
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=CONFIG["TEMPERATURE"],
                    top_p=CONFIG["TOP_P"]
                )
            )
            text_output = response.text
            if not text_output:
                raise ValueError("Received empty response payload from Gemini API.")
            
            if CONFIG["USE_CACHE"] and cache_key:
                llm_cache.set(cache_key, text_output)
            return text_output
        except Exception as e:
            last_exception = e
            print(f"⚠️ Attempt {attempt} failed with error: {e}")
            if attempt < max_retries:
                sleep_time = backoff_factor * attempt
                print(f"⏳ Retrying in {sleep_time} seconds...")
                time.sleep(sleep_time)

    # Raise explicit RuntimeError if all retries exhausted
    raise RuntimeError(f"Failed to generate LLM summary after {max_retries} attempts. Last error: {last_exception}")

## 📂 4. Load & Preprocess Exported YouTube Comment Datasets

In this section, we locate and load the exported comment dataset (`{Channel_Name}_top_comments_10_recent_videos.json` or `.csv`) generated in the previous notebook. We scan primary Google Drive storage and local fallback paths, load the structured comments, group them by video title, and compute descriptive comment volume metrics.

In [ ]:
def sanitize_filename(name: str) -> str:
    return re.sub(r'[^a-zA-Z0-9_-]', '_', name).strip('_')

safe_channel = sanitize_filename(CONFIG["CHANNEL_IDENTIFIER"])
search_directories = [ACTIVE_OUTPUT_DIR] + CONFIG["LOCAL_FALLBACK_DIRS"]

expected_json = f"{safe_channel}_top_comments_10_recent_videos.json"
expected_csv = f"{safe_channel}_top_comments_10_recent_videos.csv"

comments_data = []
loaded_file_path = None
checked_paths = []

# Search for comments dataset across primary storage and fallback directories
for directory in search_directories:
    json_path = os.path.join(directory, expected_json)
    csv_path = os.path.join(directory, expected_csv)
    checked_paths.extend([json_path, csv_path])
    
    if os.path.exists(json_path):
        with open(json_path, "r", encoding="utf-8") as f:
            comments_data = json.load(f)
        loaded_file_path = json_path
        print(f"✅ Successfully loaded JSON comments dataset from: {json_path}")
        break
    elif os.path.exists(csv_path):
        df_loaded = pd.read_csv(csv_path)
        comments_data = df_loaded.to_dict(orient="records")
        loaded_file_path = csv_path
        print(f"✅ Successfully loaded CSV comments dataset from: {csv_path}")
        break

df_comments = pd.DataFrame(comments_data)

if df_comments.empty:
    paths_list_str = "\n  - ".join(set(checked_paths))
    chan_id = CONFIG['CHANNEL_IDENTIFIER']
    err_msg = (
        f"❌ [Data Loading Error] Could not find exported comments dataset for channel '{chan_id}'.\n"
        f"Checked the following file paths:\n  - {paths_list_str}\n\n"
        f"🔍 Debugging Instructions:\n"
        f"1. Ensure you have run the preceding notebook ('youtube_video_comments.ipynb') to extract and export comments.\n"
        f"2. Confirm that Google Drive is mounted at '/content/drive' if using Drive storage.\n"
        f"3. Check that CONFIG['CHANNEL_IDENTIFIER'] in Cell 1 matches the channel identifier used during comment extraction.\n"
        f"4. Verify that '{expected_json}' or '{expected_csv}' exists in '{ACTIVE_OUTPUT_DIR}' or local fallback directories."
    )
    raise FileNotFoundError(err_msg)

# Data inspection & stats
total_comments = len(df_comments)
unique_videos = df_comments["video_title"].nunique()
avg_comments_per_vid = total_comments / max(1, unique_videos)

print(f"📊 Total Comments Loaded: {total_comments}")
print(f"📹 Unique Videos Covered: {unique_videos}")
print(f"📈 Average Comments per Video: {avg_comments_per_vid:.1f}")


## 🎯 5. Market Research Prompt Engineering & Qualitative Synthesis

In this section, we formulate structured market research prompts to extract:
1. **Per-Video Qualitative Summaries**: Core themes, top user pain points, positive resonance factors, and specific topic/feature requests.
2. **Global Channel Market Research Synthesis**: Channel-wide audience persona, dominant intent drivers, sentiment breakdown, and strategic recommendations for content creation or product offerings.

In [ ]:
def build_per_video_prompt(video_title: str, comments: List[Dict[str, Any]]) -> str:
    formatted_comments = []
    for idx, c in enumerate(comments, 1):
        text = c.get("comment_text_clean", c.get("comment_text", ""))
        likes = c.get("like_count", 0)
        formatted_comments.append(f"{idx}. [{likes} likes] {text}")
    
    comments_str = "\n".join(formatted_comments)
    
    return f"""You are an expert market research analyst specializing in digital audience sentiment and YouTube content strategy.

Analyze the following viewer comments for the video titled: "{video_title}".

Comments Dataset:
{comments_str}

Please provide a structured Market Research Synthesis in Markdown format with the following exact sections:
### Key Insights & Core Themes
- Bullet points summarizing primary viewer reactions and overarching themes.
### Audience Pain Points & Desire Drivers
- Specific challenges, confusion, or emotional needs expressed by commenters.
### Top Feature & Content Requests
- Direct requests for future video topics, tutorials, or services.
### Sentiment Breakdown
- Brief breakdown (e.g. % Positive, % Inquisitive, % Constructive/Critical).
"""

def build_global_synthesis_prompt(per_video_summaries: List[Dict[str, str]]) -> str:
    combined_summaries = []
    for idx, s in enumerate(per_video_summaries, 1):
        combined_summaries.append(f"=== Video {idx}: {s['video_title']} ===\n{s['summary']}\n")
    
    all_text = "\n".join(combined_summaries)
    
    return f"""You are a Senior Chief Strategy Officer and Market Research Director.

Below are individual qualitative summaries of user comment discussions across recent long-form videos for this channel.

Summaries Payload:
{all_text}

Synthesize these video-level findings into a comprehensive **Executive Market Research Strategy Report** for the entire channel in Markdown format.

Include the following structured sections:
# 🌐 Executive Channel Market Research Synthesis
## 1. Target Audience Profile & Key Personas
## 2. Universal Consumer Pain Points & Value Drivers
## 3. High-Demand Content & Product Opportunities
## 4. Overall Sentiment & Community Health
## 5. Strategic Recommendations for Channel Growth & Monetization
"""

# Execute Per-Video Analysis
per_video_results = []
grouped_videos = df_comments.groupby("video_title")

print(f"🚀 Beginning qualitative processing for {len(grouped_videos)} video comment clusters...")

for video_title, group in grouped_videos:
    comments_list = group.to_dict(orient="records")
    prompt = build_per_video_prompt(video_title, comments_list)
    cache_key = f"per_video_summary_{sanitize_filename(video_title)}"
    
    try:
        summary_output = call_gemini_with_retry(prompt, cache_key=cache_key)
    except Exception as e:
        print(f"⚠️ Error analyzing video '{video_title}': {e}")
        summary_output = f"*Error during LLM analysis: {e}*"
    
    per_video_results.append({
        "video_title": video_title,
        "comment_count": len(comments_list),
        "summary": summary_output
    })

# Execute Global Channel Synthesis
global_prompt = build_global_synthesis_prompt(per_video_results)
global_cache_key = f"global_synthesis_{safe_channel}"

try:
    global_market_report = call_gemini_with_retry(global_prompt, cache_key=global_cache_key)
except Exception as e:
    print(f"⚠️ Error during global channel synthesis: {e}")
    global_market_report = f"# Executive Channel Market Research Synthesis\n\n*Error generating report: {e}*"

print("✅ Market research synthesis pipeline completed successfully!")

## 📄 6. Export Market Research Report to Local & Google Drive (Markdown & JSON)

In this section, we compile all individual per-video summaries and the executive global market report into standardized output files:
1. Markdown File (`{Channel_Name}_market_research_summary.md`)
2. JSON File (`{Channel_Name}_market_research_summary.json`)

Both files are saved to Google Drive (`/content/drive/MyDrive/YouTube_Analytics/`) and mirrored to local directories.

In [ ]:
output_md_filename = f"{safe_channel}_market_research_summary.md"
output_json_filename = f"{safe_channel}_market_research_summary.json"

# Assemble full Markdown Document
md_content_blocks = [
    f"# 📊 YouTube Market Research Report: {CONFIG['CHANNEL_IDENTIFIER']}",
    f"**Generated Date**: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"**Target Model**: `{CONFIG['MODEL_NAME']}` via `google-genai` SDK",
    f"**Videos Analyzed**: {len(per_video_results)} | **Total Comments**: {len(df_comments)}",
    "\n---\n",
    global_market_report,
    "\n---\n",
    "# 📹 Individual Video Qualitative Summaries\n"
]

for idx, item in enumerate(per_video_results, 1):
    md_content_blocks.append(f"## Video {idx}: {item['video_title']}")
    md_content_blocks.append(f"*Analyzed Comment Count: {item['comment_count']}*\n")
    md_content_blocks.append(item['summary'])
    md_content_blocks.append("\n---\n")

full_md_text = "\n".join(md_content_blocks)

# Prepare JSON Payload
json_payload = {
    "channel_identifier": CONFIG["CHANNEL_IDENTIFIER"],
    "generated_at": datetime.datetime.now().isoformat(),
    "model_name": CONFIG["MODEL_NAME"],
    "total_comments_analyzed": len(df_comments),
    "videos_analyzed_count": len(per_video_results),
    "global_executive_summary": global_market_report,
    "per_video_summaries": per_video_results
}

# Export to target directories
export_directories = [ACTIVE_OUTPUT_DIR] + CONFIG["LOCAL_FALLBACK_DIRS"]
saved_paths = []

for directory in set(export_directories):
    try:
        os.makedirs(directory, exist_ok=True)
        
        # Write Markdown
        md_path = os.path.join(directory, output_md_filename)
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(full_md_text)
            
        # Write JSON
        json_path = os.path.join(directory, output_json_filename)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(json_payload, f, indent=2, ensure_ascii=False)
            
        saved_paths.append(directory)
    except Exception as e:
        print(f"⚠️ Error exporting to directory {directory}: {e}")

print(f"💾 Successfully exported Market Research files to locations: {saved_paths}")
print(f"  - Markdown Report: {output_md_filename}")
print(f"  - JSON Payload: {output_json_filename}")

## 📊 7. Analytical Metrics & Visual Analytics

To validate the dataset coverage and qualitative processing, this section generates visual analytics using Matplotlib and Seaborn:
1. **Comment Volume per Video Analyzed**: Distribution of top comments per video.
2. **Comment Character Length Distribution**: Frequency distribution of viewer comment lengths.
3. **Summary Processing Status & Token Metrics**: Qualitative summary completion status.

In compliance with self-contained figure rendering rules, figures execute `plt.show()` inline and save chart image artifacts via `plt.savefig()`.

In [ ]:
if not df_comments.empty:
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams.update({'font.size': 10, 'figure.titlesize': 14})
    
    # 1. Comment Volume per Video
    plt.figure(figsize=(12, 5))
    video_counts = df_comments["video_title"].value_counts().reset_index()
    video_counts.columns = ["video_title", "count"]
    
    ax1 = sns.barplot(data=video_counts, x="count", y="video_title", palette="viridis")
    plt.title("📊 Comment Sample Volume per Video Analyzed")
    plt.xlabel("Number of Comments Analyzed")
    plt.ylabel("Video Title")
    plt.tight_layout()
    
    fig1_path = os.path.join(ACTIVE_OUTPUT_DIR, "market_research_comment_counts.png")
    plt.savefig(fig1_path, dpi=300, bbox_inches="tight")
    plt.show()
    
    # 2. Comment Character Length Distribution
    plt.figure(figsize=(10, 5))
    if "comment_text_clean" in df_comments.columns:
        df_comments["char_len"] = df_comments["comment_text_clean"].astype(str).str.len()
    else:
        df_comments["char_len"] = df_comments["comment_text"].astype(str).str.len()
        
    sns.histplot(df_comments["char_len"], kde=True, color="teal", bins=20)
    plt.title("📏 Audience Comment Character Length Distribution")
    plt.xlabel("Character Count")
    plt.ylabel("Comment Frequency")
    plt.tight_layout()
    
    fig2_path = os.path.join(ACTIVE_OUTPUT_DIR, "market_research_char_len_dist.png")
    plt.savefig(fig2_path, dpi=300, bbox_inches="tight")
    plt.show()
    
    # 3. Qualitative Processing Summary Status
    plt.figure(figsize=(8, 4))
    status_df = pd.DataFrame(per_video_results)
    status_df["status"] = status_df["summary"].apply(lambda s: "Success" if "Error" not in s else "Failed")
    
    ax3 = sns.countplot(data=status_df, x="status", palette=["#2ecc71", "#e74c3c"])
    plt.title("🤖 LLM Market Synthesis Processing Status")
    plt.xlabel("Processing Status")
    plt.ylabel("Video Summary Count")
    plt.tight_layout()
    
    fig3_path = os.path.join(ACTIVE_OUTPUT_DIR, "market_research_llm_status.png")
    plt.savefig(fig3_path, dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("ℹ️ No comments available to render charts.")